# 잡코리아 포지션 제안 자동화 — 개발 노트북

## 개발 흐름
```
이 노트북 (셀 단위 검증)  →  jobkorea_auto.py (확정 셀렉터 반영)  →  local_server.py + admin/recruit UI
```

## 사용 방법
1. `admin/recruit/` 폴더에 `.env` 파일 준비 (`.env.example` 참고)
2. **셀을 순서대로 실행** — 브라우저가 실제로 열리므로 눈으로 보면서 셀렉터 확인
3. `# ← 교체` 주석이 있는 셀렉터 변수를 실제 동작하는 값으로 수정
4. 전체 플로우 검증 완료 → `CONFIRMED_SELECTORS` 딕셔너리를 `jobkorea_auto.py`에 반영

## 준비
```bash
pip install playwright httpx python-dotenv nest_asyncio
playwright install chromium
```

> **주의**: `slow_mo=500` 으로 설정되어 각 동작 간 0.5초 딜레이가 있습니다. 개발 완료 후 줄이세요.

In [ ]:
# 설치 확인
import sys
print(f'Python {sys.version}\n')

checks = [
    ('playwright',    'pip install playwright && playwright install chromium'),
    ('httpx',         'pip install httpx'),
    ('dotenv',        'pip install python-dotenv'),
    ('nest_asyncio',  'pip install nest_asyncio'),
]
for mod, cmd in checks:
    try:
        __import__(mod)
        print(f'  ✅ {mod}')
    except ImportError:
        print(f'  ❌ {mod} 미설치 → {cmd}')

In [ ]:
# 임포트 & 설정
import os, re, json, time
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()  # Jupyter 이벤트 루프 충돌 방지

import httpx
from dotenv import load_dotenv
from playwright.sync_api import sync_playwright, TimeoutError as PWTimeout

# .env 로드 (노트북과 같은 폴더의 .env)
load_dotenv(override=True)

JOBKOREA_ID  = os.getenv('JOBKOREA_ID', '')
JOBKOREA_PW  = os.getenv('JOBKOREA_PW', '')
SUPABASE_URL = os.getenv('SUPABASE_URL', '')
SUPABASE_KEY = os.getenv('SUPABASE_SERVICE_KEY', '')
PROFILE_DIR  = os.getenv('PROFILE_DIR', str(Path.home() / '.jobkorea_profile'))

TALENT_URL = 'https://www.jobkorea.co.kr/Recruit/Co_Read/C/personsearch'
LOGIN_URL  = 'https://www.jobkorea.co.kr/User/LogOn/LogOn'

assert JOBKOREA_ID, '❌ .env에 JOBKOREA_ID 없음'
assert JOBKOREA_PW, '❌ .env에 JOBKOREA_PW 없음'
print(f'✅ ID: {JOBKOREA_ID[:3]}***')
print(f'   프로필 디렉토리: {PROFILE_DIR}')
print(f'   Supabase: {"연결됨" if SUPABASE_URL else "미설정 (발송 기록 저장 불가)"}')

---
## STEP 0: 브라우저 시작 / 종료

- 브라우저는 **한 번만 시작** — 이후 셀은 `JK['page']`로 재사용
- `--user-data-dir` 로 세션이 유지되므로 로그인은 첫 실행 후 자동 재사용됨
- 브라우저를 직접 보면서 각 단계를 확인하세요 (`headless=False`)

In [ ]:
# ▶ 브라우저 시작 (이 셀은 1회만 실행)
# 이미 열려 있으면 기존 세션 재사용

if 'JK' not in globals() or not globals().get('JK', {}).get('alive'):
    _pw = sync_playwright().start()
    _ctx = _pw.chromium.launch_persistent_context(
        user_data_dir=PROFILE_DIR,
        headless=False,   # 반드시 False — 직접 보면서 개발
        slow_mo=500,      # 동작 간 딜레이(ms) — 확인용, 완료 후 줄임
        args=[
            '--disable-blink-features=AutomationControlled',
            '--no-sandbox',
        ],
    )
    _page = _ctx.pages[0] if _ctx.pages else _ctx.new_page()
    JK = {'pw': _pw, 'ctx': _ctx, 'page': _page, 'alive': True}
    print('✅ 브라우저 시작됨')
else:
    print('ℹ️  기존 브라우저 재사용')

page = JK['page']
print(f'현재 URL: {page.url}')

In [ ]:
# 로그인 상태 확인
page = JK['page']
page.goto(TALENT_URL, wait_until='domcontentloaded')
time.sleep(2)

if TALENT_URL.split('?')[0] in page.url:
    print('✅ 로그인 상태 유지됨 — 인재검색 페이지 진입 성공')
    print('   다음 셀(로그인 폼 탐색)은 건너뛰고 STEP 1 부터 진행')
else:
    print(f'🔑 세션 없음 (현재: {page.url})')
    print('   아래 셀에서 로그인 폼 셀렉터를 확인하세요')
    page.goto(LOGIN_URL, wait_until='domcontentloaded')
    time.sleep(1)
    print(f'   로그인 페이지: {page.url}')

---
## 로그인 폼 셀렉터 확인 (세션 없을 때만)

세션이 이미 있으면 이 섹션 건너뛰기

In [ ]:
# 로그인 폼 셀렉터 탐색
page = JK['page']
print(f'현재 URL: {page.url}\n')

FORM_CHECKS = {
    '아이디 입력': ['input[name="id"]', 'input[name="uId"]', '#id', 'input[type="text"]'],
    '비밀번호':   ['input[name="pw"]', 'input[name="pwd"]', '#pw', 'input[type="password"]'],
    '로그인 버튼': ['button[type="submit"]', 'input[type="submit"]', '.btn-login', '#btnLogin'],
}

for label, sels in FORM_CHECKS.items():
    print(f'[{label}]')
    for sel in sels:
        try:
            el = page.locator(sel).first
            if el.is_visible(timeout=400):
                print(f'  ✅ {sel}')
                break
            else:
                print(f'  ⬜ {sel} (DOM에 있지만 숨김)')
        except:
            print(f'  ❌ {sel}')

In [ ]:
# 로그인 실행
# ★ 위 셀에서 확인한 셀렉터로 교체
SEL_ID  = 'input[name="id"]'       # ← 교체
SEL_PW  = 'input[name="pw"]'       # ← 교체
SEL_BTN = 'button[type="submit"]'  # ← 교체

page = JK['page']
page.fill(SEL_ID, JOBKOREA_ID)
page.fill(SEL_PW, JOBKOREA_PW)
page.click(SEL_BTN)
time.sleep(3)

print(f'로그인 후 URL: {page.url}')
if 'login' in page.url.lower() or 'logon' in page.url.lower():
    print('❌ 로그인 실패 — 셀렉터 또는 ID/PW 확인 필요')
else:
    print('✅ 로그인 성공')
    page.goto(TALENT_URL, wait_until='domcontentloaded')
    time.sleep(2)
    print(f'인재검색 페이지: {page.url}')

---
## STEP 1: 인재검색 — 검색창 & 결과 파싱

각 셀에서 `✅` 표시된 셀렉터를 `# ← 교체` 위치에 복사

In [ ]:
# 인재검색 페이지 이동 + 검색창 셀렉터 탐색
page = JK['page']
page.goto(TALENT_URL, wait_until='domcontentloaded')
time.sleep(2)
print(f'URL: {page.url}')
print(f'타이틀: {page.title()}\n')

SEARCH_BOX_SELS = [
    'input[name="stext"]', '#searchText', 'input[placeholder*="검색"]',
    'input[type="search"]', '.search-input', '#keyword', 'input[name="query"]',
]

print('[검색창 셀렉터]')
for sel in SEARCH_BOX_SELS:
    try:
        el = page.locator(sel).first
        if el.is_visible(timeout=400):
            ph = el.get_attribute('placeholder') or ''
            print(f'  ✅ {sel}  placeholder="{ph}"')
        else:
            print(f'  ⬜ {sel}')
    except:
        print(f'  ❌ {sel}')

In [ ]:
# 키워드 검색 실행 + 후보자 카드 컨테이너 탐색
KEYWORD = '법인영업'  # 테스트 키워드
SEL_SEARCH_BOX = 'input[name="stext"]'  # ← 교체

page = JK['page']
box = page.locator(SEL_SEARCH_BOX).first
box.fill(KEYWORD)
box.press('Enter')
time.sleep(3)
print(f'검색 후 URL: {page.url}\n')

CARD_SELS = [
    '.resumeList li', '.person-item', 'li.people-item',
    '.userinfo-wrap', 'tr.resume-row', '.resume-item',
    '.person-list-item', '.search-list li', '.list-item',
]

print('[후보자 카드 컨테이너]')
for sel in CARD_SELS:
    try:
        count = page.locator(sel).count()
        marker = '✅' if count > 0 else '❌'
        print(f'  {marker} {sel} → {count}개')
    except Exception as e:
        print(f'  ⚠ {sel} → {e}')

In [ ]:
# 첫 번째 카드 내부 구조 탐색
SEL_CARD = '.resumeList li'  # ← 교체

page = JK['page']
cards = page.locator(SEL_CARD).all()
print(f'카드 수: {len(cards)}\n')

if not cards:
    print('❌ 카드 없음 — SEL_CARD 셀렉터를 수정하세요')
else:
    card = cards[0]

    print('=== 카드 전체 텍스트 ===')
    try:
        print(card.inner_text(timeout=1000)[:400])
    except: pass

    # 이름
    NAME_SELS = ['.name', '.userName', 'strong.name', '.user-name', 'td.name', 'a.name', 'h3', 'h4']
    print('\n[이름 셀렉터]')
    for sel in NAME_SELS:
        try:
            t = card.locator(sel).first.inner_text(timeout=300).strip()
            print(f'  {"✅" if t else "⬜"} {sel} → "{t[:30]}"')
        except: print(f'  ❌ {sel}')

    # 경력
    CAREER_SELS = ['.career', '.userCareer', '.career-info', 'td.career', '.exp', '.career-txt', '.career-summary']
    print('\n[경력 셀렉터]')
    for sel in CAREER_SELS:
        try:
            t = card.locator(sel).first.inner_text(timeout=300).strip()
            print(f'  {"✅" if t else "⬜"} {sel} → "{t[:50]}"')
        except: print(f'  ❌ {sel}')

    # 후보자 ID
    ID_ATTRS = ['data-uIdx', 'data-idx', 'data-person-idx', 'data-id', 'data-seq']
    print('\n[ID 속성 (카드 자체)]')
    for attr in ID_ATTRS:
        val = card.get_attribute(attr)
        print(f'  {"✅" if val else "❌"} {attr} = "{val}"')

    # href 에서 ID 추출
    print('\n[링크에서 ID 추출]')
    try:
        links = card.locator('a').all()
        for link in links[:5]:
            href = link.get_attribute('href') or ''
            if 'idx' in href.lower():
                m = re.search(r'[uU]?[Ii]dx=(\d+)', href)
                print(f'  링크: {href[:80]}  →  ID={m.group(1) if m else "없음"}')
    except: pass

In [ ]:
# ★ 위 셀 결과로 셀렉터 확정 후 실행
SEL_CARD   = '.resumeList li'   # ← 교체
SEL_NAME   = '.name'            # ← 교체
SEL_CAREER = '.career'          # ← 교체
ID_ATTR    = 'data-uIdx'        # ← 교체 (없으면 'href' 방식 사용)

page = JK['page']
cards = page.locator(SEL_CARD).all()
print(f'총 {len(cards)}개 카드 파싱\n')

candidates = []
for i, card in enumerate(cards):
    try:
        name = card.locator(SEL_NAME).first.inner_text(timeout=500).strip()
        career = ''
        try: career = card.locator(SEL_CAREER).first.inner_text(timeout=300).strip()
        except: pass

        cid = card.get_attribute(ID_ATTR) or ''
        if not cid:
            try:
                href = card.locator('a[href*="idx"]').first.get_attribute('href', timeout=300) or ''
                m = re.search(r'[uU]?[Ii]dx=(\d+)', href)
                if m: cid = m.group(1)
            except: pass

        ok = bool(name and cid)
        marker = '✅' if ok else '⚠'
        print(f'  {marker} [{i+1}] {name or "(이름없음)"} | {career[:35]} | ID={cid or "없음"}')
        if ok:
            candidates.append({'candidate_id': cid, 'name': name, 'career': career})
    except Exception as e:
        print(f'  ❌ [{i+1}] 오류: {e}')

print(f'\n✅ 파싱 완료: {len(candidates)}명')
JK['candidates'] = candidates  # 이후 셀에서 재사용

---
## STEP 2: 포지션 제안 — 버튼 & 팝업 셀렉터

- 잡코리아 사이트 내 포지션 제안 기능 사용 (이메일 직접 발송 아님)
- 제안 버튼 → 팝업 → 메시지 입력 → 발송 순서

In [ ]:
# 포지션 제안 버튼 셀렉터 탐색
SEL_CARD = '.resumeList li'  # ← 교체

page = JK['page']
cards = page.locator(SEL_CARD).all()

if not cards:
    print('❌ 카드 없음 — 먼저 검색 셀을 실행하세요')
else:
    card = cards[0]
    print(f'첫 번째 카드 내 버튼/링크 전체 목록:\n')

    # 카드 내 모든 버튼 + 링크 텍스트 출력
    for el in card.locator('button, a').all():
        try:
            text = el.inner_text(timeout=200).strip()
            cls  = el.get_attribute('class') or ''
            href = el.get_attribute('href') or ''
            if text:
                print(f'  [{el.evaluate("e => e.tagName").lower()}] "{text}"  class="{cls[:40]}"  href="{href[:40]}"')
        except: pass

    # 제안 버튼 후보 셀렉터
    PROP_BTN_SELS = [
        'button:has-text("포지션")', 'button:has-text("제안")', 'button:has-text("입사제안")',
        'a:has-text("포지션")', 'a:has-text("제안")', 'a:has-text("입사제안")',
        '.btn-proposal', '.offer-btn', '[class*="proposal"]', '[class*="offer"]',
    ]
    print('\n[제안 버튼 셀렉터 탐색]')
    for sel in PROP_BTN_SELS:
        try:
            count = card.locator(sel).count()
            if count:
                t = card.locator(sel).first.inner_text(timeout=200).strip()
                print(f'  ✅ {sel} → "{t}"')
        except: pass

In [ ]:
# 제안 버튼 클릭 → 팝업/모달 내 셀렉터 탐색
# ★ 실제로 팝업이 열립니다 (발송은 아직 안 함)

SEL_CARD     = '.resumeList li'               # ← 교체
SEL_PROP_BTN = 'button:has-text("포지션")'    # ← 교체

page = JK['page']
cards = page.locator(SEL_CARD).all()

if not cards:
    print('❌ 카드 없음')
else:
    try:
        btn = cards[0].locator(SEL_PROP_BTN).first
        btn.click()
        time.sleep(2)
        print(f'✅ 클릭 완료 — 팝업 여부: {page.url}\n')

        # 메시지 입력창 탐색
        MSG_SELS = [
            'textarea', 'textarea.message', '#proposalMsg',
            'textarea[name="message"]', '.modal textarea', '.popup textarea', 'dialog textarea',
        ]
        print('[메시지 입력창]')
        for sel in MSG_SELS:
            try:
                count = page.locator(sel).count()
                if count:
                    vis = page.locator(sel).first.is_visible(timeout=300)
                    print(f'  {"✅" if vis else "⬜"} {sel} → {count}개 ({"보임" if vis else "숨김"})')
            except: pass

        # 발송 버튼 탐색
        SEND_SELS = [
            'button:has-text("발송")', 'button:has-text("제안하기")', 'button:has-text("확인")',
            'input[value="발송"]', '.btn-send', '.modal button[type="submit"]', 'dialog button',
        ]
        print('\n[발송 버튼]')
        for sel in SEND_SELS:
            try:
                count = page.locator(sel).count()
                if count:
                    t = page.locator(sel).first.inner_text(timeout=200).strip()
                    vis = page.locator(sel).first.is_visible(timeout=200)
                    print(f'  {"✅" if vis else "⬜"} {sel} → "{t}"')
            except: pass

        # 팝업 내 모든 버튼
        print('\n[팝업 내 모든 버튼]')
        for sel in ['.modal button', '.popup button', 'dialog button', '[role="dialog"] button']:
            try:
                btns = page.locator(sel).all()
                for b in btns:
                    t = b.inner_text(timeout=200).strip()
                    cls = b.get_attribute('class') or ''
                    if t: print(f'    [{sel}] "{t}"  class="{cls[:40]}"')
            except: pass

    except Exception as e:
        print(f'❌ 실패: {e}')

In [ ]:
# 단건 발송 테스트
# ★ 실제 발송됩니다 — 팝업이 열려있는 상태에서 실행

SEL_MSG_INPUT = 'textarea'                      # ← 교체
SEL_SEND_BTN  = 'button:has-text("발송")'       # ← 교체

PROPOSAL_MSG = """안녕하세요!

삼성생명(주)에서 채용의뢰를 받은 용산HR 이인성 팀장입니다.

현재 삼성생명 법인영업(GFC) 포지션을 모집 중입니다.
기업 CEO를 대상으로 세무·재무·리스크 컨설팅을 담당하는 전문직으로,
입사 후 13개월간 세무사·회계사·노무사 강의와 1:1 멘토링을 제공합니다.

10분 정도 편하신 시간에 통화 가능할까요?

이인성 팀장 | 삼성생명 용산HR"""

page = JK['page']

try:
    msg_box = page.locator(SEL_MSG_INPUT).first
    msg_box.clear()
    msg_box.fill(PROPOSAL_MSG)
    print(f'✅ 메시지 입력 완료 ({len(PROPOSAL_MSG)}자)')
    time.sleep(1)

    send_btn = page.locator(SEL_SEND_BTN).first
    send_btn.click()
    time.sleep(2)

    print('✅ 발송 버튼 클릭 — 브라우저에서 결과 확인')
    print(f'현재 URL: {page.url}')
except Exception as e:
    print(f'❌ 실패: {e}')

---
## STEP 3: Supabase 연동 — 중복 방지 & 발송 기록

In [ ]:
# Supabase 조회 테스트 — 기발송 목록
if not SUPABASE_URL or not SUPABASE_KEY:
    print('⚠ SUPABASE_URL / SUPABASE_SERVICE_KEY 미설정 — .env 확인')
else:
    headers = {
        'apikey':        SUPABASE_KEY,
        'Authorization': f'Bearer {SUPABASE_KEY}',
        'Content-Type':  'application/json',
    }
    with httpx.Client(timeout=10) as c:
        r = c.get(
            f'{SUPABASE_URL}/rest/v1/jobkorea_proposals?select=candidate_id,name,sent_at&order=sent_at.desc&limit=10',
            headers=headers,
        )
    if r.status_code == 200:
        rows = r.json()
        print(f'✅ 연결 성공 — 최근 발송 {len(rows)}건')
        for row in rows:
            print(f'   {row["name"]} | {row["sent_at"][:10]} | ID={row["candidate_id"]}')
    else:
        print(f'❌ 오류 {r.status_code}: {r.text}')
        print('   → jobkorea_setup.sql 실행 여부 확인 (Supabase SQL Editor)')

In [ ]:
# Supabase 저장 테스트 (더미 데이터)
if not SUPABASE_URL or not SUPABASE_KEY:
    print('⚠ Supabase 미설정')
else:
    headers = {
        'apikey':        SUPABASE_KEY,
        'Authorization': f'Bearer {SUPABASE_KEY}',
        'Content-Type':  'application/json',
        'Prefer':        'resolution=merge-duplicates,return=minimal',
    }
    test_row = {
        'candidate_id': 'NOTEBOOK_TEST_001',
        'name':         '노트북테스트',
        'career':       '테스트 경력',
        'keyword':      'test',
        'status':       'skipped',
        'notes':        'notebook 개발 테스트 기록 — 삭제해도 됩니다',
    }
    with httpx.Client(timeout=10) as c:
        r = c.post(f'{SUPABASE_URL}/rest/v1/jobkorea_proposals', headers=headers, json=test_row)

    if r.status_code in (200, 201, 204):
        print('✅ 저장 성공')
        # 확인
        with httpx.Client(timeout=10) as c:
            r2 = c.get(
                f'{SUPABASE_URL}/rest/v1/jobkorea_proposals?candidate_id=eq.NOTEBOOK_TEST_001&select=*',
                headers=headers,
            )
        print(f'저장된 레코드: {r2.json()}')
    else:
        print(f'❌ 저장 실패 {r.status_code}: {r.text}')

---
## STEP 4: 전체 플로우 통합 테스트

**이 섹션은 STEP 1~3 셀렉터가 모두 확정된 후에 실행하세요.**

1. 아래 `CONFIRMED_SELECTORS` 딕셔너리에 검증된 셀렉터 채우기
2. `DRY_RUN = True` 로 먼저 실행 (발송 없이 수집만)
3. 문제 없으면 `DRY_RUN = False` 로 소규모 테스트 (3~5명)
4. 검증 완료 → `jobkorea_auto.py` 에 반영

In [ ]:
# ★★★ 검증된 셀렉터 모음 — 여기만 수정하면 됩니다 ★★★
# 이 딕셔너리가 완성되면 jobkorea_auto.py 에 그대로 복사

CONFIRMED_SELECTORS = {
    # 로그인
    'login_id':     'input[name="id"]',       # ← 교체
    'login_pw':     'input[name="pw"]',       # ← 교체
    'login_btn':    'button[type="submit"]',  # ← 교체

    # 검색
    'search_box':   'input[name="stext"]',    # ← 교체

    # 후보자 카드
    'card':         '.resumeList li',          # ← 교체
    'name':         '.name',                   # ← 교체
    'career':       '.career',                 # ← 교체
    'id_attr':      'data-uIdx',              # ← 교체 (속성명 또는 'href')

    # 포지션 제안
    'proposal_btn': 'button:has-text("포지션")',  # ← 교체
    'msg_input':    'textarea',                   # ← 교체
    'send_btn':     'button:has-text("발송")',    # ← 교체

    # 페이지네이션
    'next_page':    'a.next, .paging-next',   # ← 교체
}

print('현재 셀렉터 설정:')
for k, v in CONFIRMED_SELECTORS.items():
    print(f'  {k:15} = "{v}"')

In [ ]:
# 전체 플로우 실행
# DRY_RUN = True  → 후보자 수집만, 실제 발송 없음
# DRY_RUN = False → 실제 발송 (소규모 테스트)

DRY_RUN   = True        # ← 검증 완료 후 False로 변경
KEYWORD   = '법인영업'  # 검색 키워드
MAX_PAGES = 1           # 페이지 수 (테스트는 1)
MAX_SEND  = 3           # 최대 발송 수 (테스트용)

S = CONFIRMED_SELECTORS  # 단축 참조

PROPOSAL_MSG = """안녕하세요, {name}님!

삼성생명(주)에서 채용의뢰를 받은 용산HR 이인성 팀장입니다.

{name}님의 {career_short} 경험이 매우 인상적이어서 연락드립니다.

현재 삼성생명 법인영업(GFC) 포지션을 모집 중입니다.
기업 CEO를 대상으로 세무·재무·리스크 컨설팅을 담당하는 전문직으로,
입사 후 13개월간 세무사·회계사·노무사 강의와 1:1 멘토링을 제공합니다.

10분 정도 편하신 시간에 통화 가능할까요?

이인성 팀장 | 삼성생명 용산HR"""

# Supabase 기발송 목록
already_sent = set()
if SUPABASE_URL and SUPABASE_KEY:
    hdrs = {'apikey': SUPABASE_KEY, 'Authorization': f'Bearer {SUPABASE_KEY}', 'Content-Type': 'application/json'}
    with httpx.Client(timeout=10) as c:
        r = c.get(f'{SUPABASE_URL}/rest/v1/jobkorea_proposals?select=candidate_id&limit=10000', headers=hdrs)
        if r.status_code == 200:
            already_sent = {row['candidate_id'] for row in r.json()}
    print(f'기발송 {len(already_sent)}건 로드')

# 검색
page = JK['page']
page.goto(TALENT_URL, wait_until='domcontentloaded')
time.sleep(2)
box = page.locator(S['search_box']).first
box.fill(KEYWORD); box.press('Enter')
time.sleep(3)

sent = skipped = errors = 0

for pg in range(MAX_PAGES):
    cards = page.locator(S['card']).all()
    print(f'\n페이지 {pg+1}: {len(cards)}개 카드')

    for card in cards:
        if sent >= MAX_SEND:
            print(f'\n최대 발송 수 {MAX_SEND}건 도달 — 중단'); break
        try:
            name = card.locator(S['name']).first.inner_text(timeout=500).strip()
            career = ''
            try: career = card.locator(S['career']).first.inner_text(timeout=300).strip()
            except: pass

            cid = card.get_attribute(S['id_attr']) or ''
            if not cid:
                try:
                    href = card.locator('a[href*="idx"]').first.get_attribute('href', timeout=300) or ''
                    m = re.search(r'[uU]?[Ii]dx=(\d+)', href)
                    if m: cid = m.group(1)
                except: pass

            if not name or not cid: continue

            if cid in already_sent:
                print(f'  [스킵] {name} — 기발송')
                skipped += 1; continue

            if DRY_RUN:
                print(f'  [DRY] {name} ({cid}) — 발송 생략')
                sent += 1
                continue

            # 실제 발송
            msg = PROPOSAL_MSG.format(name=name, career_short=career[:20] if career else '풍부한')
            card.locator(S['proposal_btn']).first.click(); time.sleep(1.5)
            page.locator(S['msg_input']).first.fill(msg)
            page.locator(S['send_btn']).first.click(); time.sleep(2)
            print(f'  ✅ 발송: {name} ({cid})')

            if SUPABASE_URL and SUPABASE_KEY:
                row = {'candidate_id': cid, 'name': name, 'career': career, 'keyword': KEYWORD, 'status': 'sent'}
                hdrs2 = {**hdrs, 'Prefer': 'resolution=merge-duplicates,return=minimal'}
                with httpx.Client(timeout=10) as c:
                    c.post(f'{SUPABASE_URL}/rest/v1/jobkorea_proposals', headers=hdrs2, json=row)
            already_sent.add(cid)
            sent += 1
            time.sleep(2)

        except Exception as e:
            print(f'  ❌ 오류: {e}')
            errors += 1

    # 다음 페이지
    if pg < MAX_PAGES - 1:
        try:
            nxt = page.locator(S['next_page']).first
            if nxt.is_visible(timeout=1000): nxt.click(); time.sleep(2)
            else: break
        except: break

print(f'\n=== 완료 === 발송 {sent} · 스킵 {skipped} · 오류 {errors}')
if DRY_RUN:
    print('DRY_RUN = True 상태 — 실제 발송 없음. 확인 후 False로 변경하세요.')

---
## jobkorea_auto.py 통합 가이드

전체 플로우 검증이 완료되면 아래 항목을 `jobkorea_auto.py`에 반영합니다.

### 1. 셀렉터 교체
`CONFIRMED_SELECTORS` 딕셔너리 값을 `jobkorea_auto.py` 내 해당 위치에 복사:

| 노트북 키 | jobkorea_auto.py 위치 |
|---|---|
| `search_box` | `collect_candidates()` → 검색창 `for sel in [...]` |
| `card` | `collect_candidates()` → 카드 `for sel in [...]` |
| `name` | 이름 셀렉터 목록 |
| `career` | 경력 셀렉터 목록 |
| `id_attr` | `data-uIdx` 등 속성 목록 |
| `proposal_btn` | `send_proposal()` → 버튼 탐색 목록 |
| `msg_input` | `send_proposal()` → 메시지 입력창 목록 |
| `send_btn` | `send_proposal()` → 발송 버튼 목록 |

### 2. 검증 완료 체크리스트
- [ ] 로그인 세션 유지 확인
- [ ] 후보자 카드 파싱 (이름, 경력, ID 모두 추출)
- [ ] 포지션 제안 팝업 열기
- [ ] 메시지 입력 + 발송
- [ ] Supabase 발송 기록 저장
- [ ] 중복 발송 방지 동작 확인
- [ ] `DRY_RUN=False` 로 실제 5명 발송 테스트

In [ ]:
# ■ 브라우저 종료 (세션은 PROFILE_DIR 에 저장됨)
# 세션이 저장되므로 다음 실행 시 로그인 불필요

if 'JK' in globals() and JK.get('alive'):
    JK['ctx'].close()
    JK['pw'].stop()
    JK['alive'] = False
    print(f'✅ 브라우저 종료 완료')
    print(f'   세션 저장 위치: {PROFILE_DIR}')
else:
    print('ℹ️  브라우저가 이미 종료되었습니다')